In [2]:
# Importacoes e Inicializacao da SparkSession
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Formatos-Spark-Avro")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.spark:spark-avro_2.12:3.5.0")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "root-minio")
    .config("spark.hadoop.fs.s3a.secret.key", "root12345678")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

print(f"SparkSession ativa. Versao: {spark.version}")

SparkSession ativa. Versao: 3.5.0


In [3]:
# Extracao (Extract) do PostgreSQL via JDBC
print("Extraindo dados para serializacao binaria por linha...")

df_financeiro = (spark.read.jdbc(
    url="jdbc:postgresql://postgres:5432/postgres",
    table="transacoes_financeiras",
    properties={"user": "postgres", "password": "senha123", "driver": "org.postgresql.Driver"}
))

df_financeiro.printSchema()

Extraindo dados para serializacao binaria por linha...
root
 |-- id_transacao: string (nullable = true)
 |-- conta_origem: string (nullable = true)
 |-- conta_destino: string (nullable = true)
 |-- tipo_movimento: string (nullable = true)
 |-- valor: decimal(15,4) (nullable = true)
 |-- moeda: string (nullable = true)
 |-- data_transacao: timestamp (nullable = true)
 |-- hash_auditoria: string (nullable = true)
 |-- data_criacao: timestamp (nullable = true)



In [4]:
# Carga (Load) para o MinIO em formato Avro
bucket_name = "raw"
destino_s3 = f"s3a://{bucket_name}/spark_jobs/ledger_avro/"

print(f"Gravando dados em formato Avro no Object Storage: {destino_s3}")

# O Spark mapeia as tipagens do DataFrame e constroi o Schema Avro de forma automatizada nos metadados
df_financeiro.write.mode("overwrite").format("avro").save(destino_s3)

print("Escrita do Avro concluida com sucesso no destino.")

Gravando dados em formato Avro no Object Storage: s3a://raw/spark_jobs/ledger_avro/
Escrita do Avro concluida com sucesso no destino.


In [5]:
# Auditoria e Validação dos Dados Gravados
print("Lendo arquivos Avro de volta do Data Lake para validacao...")

# Necessario especificar o formato 'avro' adicionado via dependência externa
df_validacao = spark.read.format("avro").load(destino_s3)

print(f"Quantidade total de registros localizados: {df_validacao.count()}")
df_validacao.show(5, truncate=False)

Lendo arquivos Avro de volta do Data Lake para validacao...
Quantidade total de registros localizados: 20000
+------------------------------------+------------------+------------------+--------------+----------+-----+-------------------+----------------------------------------------------------------+-------------------+
|id_transacao                        |conta_origem      |conta_destino     |tipo_movimento|valor     |moeda|data_transacao     |hash_auditoria                                                  |data_criacao       |
+------------------------------------+------------------+------------------+--------------+----------+-----+-------------------+----------------------------------------------------------------+-------------------+
|e9b4ba43-ef07-4d76-8436-4c829966c49b|IYXG33090884540679|NMIZ81024723375965|DEBITO        |14236.8874|BRL  |2026-04-16 23:53:15|e23c1a3adb95d63fdc32c8eedd0f87db6fd050073b12e7359e4af8d17677ab40|2026-04-16 23:53:15|
|000df2fb-d92d-4bef-b71c-6d9094aed7

In [6]:
# Encerramento da Sessão e Limpeza de Recursos
print("Encerrando a SparkSession e liberando os recursos da JVM...")
spark.stop()
print("Sessao fechada com seguranca.")

Encerrando a SparkSession e liberando os recursos da JVM...
Sessao fechada com seguranca.


### Análise de Arquitetura: O Formato Avro no Apache Spark

**Características Principais:** Formato binário, autodescritivo e estritamente orientado a linhas (Row-based).

#### Vantagens
* **Abstração e Governança:** O Spark consegue inferir os tipos do DataFrame e embutir o contrato de dados (Schema JSON) diretamente no cabeçalho do arquivo binário gerado.
* **Velocidade de Escrita:** Como a gravação é feita de maneira sequencial linha por linha, o overhead computacional para persistir os dados é baixíssimo, tornando o processo de escrita muito mais rápido que o do Parquet.
* **Evolução de Esquema Segura:** É o formato mais robusto para gerenciar alterações na estrutura das tabelas ao longo do tempo (Schema Evolution), suportando adição ou remoção de colunas sem corromper a leitura de arquivos legados.

#### Desvantagens
* **Ineficiente para Consultas Analíticas (OLAP):** Se o objetivo for executar uma agregação matemática (como a soma de uma coluna financeira), o motor de busca será obrigado a ler e carregar na memória a linha inteira de todos os registros para extrair o valor daquela coluna específica.

#### Casos de Uso no Mercado
* Formato padrão da indústria para pipelines de dados em tempo real e arquiteturas de streaming baseadas em mensageria (como Apache Kafka e AWS Kinesis).
* Camadas de Landings Zones ou áreas de Staging de alta velocidade, onde o pipeline de ingestão precisa descarregar os dados transacionais o mais rápido possível no Data Lake sem causar gargalos de processamento.